In [31]:
import numpy as np
import os
import pandas as pd
import duckdb
from sqlalchemy import create_engine


In [32]:
engine = create_engine(os.getenv("DATABASE_URL"))

conn = engine.raw_connection()
try:
    df = pd.read_sql(
        "SELECT form_id,contract_num,year_stamp_holo_dc FROM jjm_customer_loan WHERE brand = 'HERMES' and type = 'Bag' and status =1",
        conn
    )
finally:
    conn.close() 


/tmp/ipykernel_683/3258780893.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


In [33]:
import re
import pandas as pd

_sq_cir_re = re.compile(r"\b(square|circle)\b", flags=re.IGNORECASE)
_year_re = re.compile(r"\b(19\d{2}|20\d{2})\b")

def extract_year_mapping(text: object) -> object:
    if pd.isna(text):
        return pd.NA
    m = _year_re.search(str(text))
    return m.group(1) if m else pd.NA


def extract_year_letter(text: object) -> object:
    if pd.isna(text):
        return pd.NA

    s = str(text).strip()
    if not s:
        return pd.NA

    s_norm = re.sub(r"\s+", " ", s)

    # -------------------------------
    # Case 2) contains Square / Circle
    # -------------------------------
    m = _sq_cir_re.search(s_norm)
    if m:
        shape_raw = m.group(1).lower()
        shape = "Square" if shape_raw == "square" else "Circle"

        left = s_norm[:m.start()].strip()
        right = s_norm[m.end():].strip()

        # prefer letter BEFORE shape
        before = re.findall(r"\b([A-Za-z])\b", left)
        if before:
            return f"{before[-1].upper()} {shape}"

        # if shape is at beginning → take letter AFTER
        after = re.search(r"\b([A-Za-z])\b", right)
        if after:
            return f"{after.group(1).upper()} {shape}"

        return pd.NA

    # -------------------------------
    # Case 1) no Square / Circle
    # -------------------------------
    for i, ch in enumerate(s_norm):
        if "A" <= ch <= "Z":
            prev_is_digit = (i > 0 and s_norm[i - 1].isdigit())
            next_is_digit = (i + 1 < len(s_norm) and s_norm[i + 1].isdigit())
            if not prev_is_digit and not next_is_digit:
                return ch

    return pd.NA


# create columns
df["year_mapping"] = df["year_stamp_holo_dc"].apply(extract_year_mapping)
df["year_letter"] = df["year_stamp_holo_dc"].apply(extract_year_letter)

In [34]:
# WITHOUT SHAPE (1945–1970)
WITHOUT_SHAPE_MAP = {
    "A": 1945, "B": 1946, "C": 1947, "D": 1948, "E": 1949,
    "F": 1950, "G": 1951, "H": 1952, "I": 1953, "J": 1954,
    "K": 1955, "L": 1956, "M": 1957, "N": 1958, "O": 1959,
    "P": 1960, "Q": 1961, "R": 1962, "S": 1963, "T": 1964,
    "U": 1965, "V": 1966, "W": 1967, "X": 1968, "Y": 1969,
    "Z": 1970,
}

# CIRCLE SHAPE (1971–1996)
CIRCLE_MAP = {
    "A": 1971, "B": 1972, "C": 1973, "D": 1974, "E": 1975,
    "F": 1976, "G": 1977, "H": 1978, "I": 1979, "J": 1980,
    "K": 1981, "L": 1982, "M": 1983, "N": 1984, "O": 1985,
    "P": 1986, "Q": 1987, "R": 1988, "S": 1989, "T": 1990,
    "U": 1991, "V": 1992, "W": 1993, "X": 1994, "Y": 1995,
    "Z": 1996,
}

# SQUARE SHAPE (1997–2014)
SQUARE_MAP = {
    "A": 1997, "B": 1998, "C": 1999, "D": 2000, "E": 2001,
    "F": 2002, "G": 2003, "H": 2004, "I": 2005, "J": 2006,
    "K": 2007, "L": 2008, "M": 2009, "N": 2010, "O": 2011,
    "P": 2012, "Q": 2013, "R": 2014,
}
INTERIOR_MAP = {
    "T": 2015,
    "X": 2016,
    "A": 2017,
    "C": 2018,
    "D": 2019,
    "Y": 2020,
    "Z": 2021,
    "U": 2022,
    "B": 2023,
    "W": 2024,
    "K": 2025,
}
# letters that exist in Square/Circle regimes (potentially missing the shape in raw data)
SQUARE_LETTERS = set(SQUARE_MAP.keys())     # A..R
CIRCLE_LETTERS = set(CIRCLE_MAP.keys())     # A..Z
SHAPE_LETTERS = SQUARE_LETTERS | CIRCLE_LETTERS


In [35]:
import pandas as pd

def map_year_from_letter(row):
    """
    Map year_letter -> year_mapping

    เงื่อนไข:
    - ทำเฉพาะ row ที่ year_letter มีค่า
    - และ year_mapping ยังเป็น NaN

    ผลลัพธ์:
    - map ได้ -> ปี (int)
    - map ไม่ได้ทุกกรณี -> pd.NA
    """
    # ถ้ามี year_mapping อยู่แล้ว หรือไม่มี year_letter -> ไม่แตะ
    if pd.notna(row["year_mapping"]) or pd.isna(row["year_letter"]):
        return row["year_mapping"]

    yl = str(row["year_letter"]).strip()

    # -------- Square --------
    if yl.endswith("Square"):
        letter = yl.replace("Square", "").strip()
        return SQUARE_MAP.get(letter, pd.NA)

    # -------- Circle --------
    if yl.endswith("Circle"):
        letter = yl.replace("Circle", "").strip()
        return CIRCLE_MAP.get(letter, pd.NA)

    # -------- Without shape (single letter) --------
    if len(yl) == 1 and yl.isalpha():
        # interior overrides
        if yl in INTERIOR_MAP:
            return INTERIOR_MAP[yl]

        # ambiguous with Square/Circle (missing shape) -> NULL
        if yl in SHAPE_LETTERS:
            return pd.NA

        # safe old-without-shape
        return WITHOUT_SHAPE_MAP.get(yl, pd.NA)

    # -------- Everything else --------
    return pd.NA


df["year_mapping"] = df.apply(map_year_from_letter, axis=1)


In [38]:
df[df['year_mapping'].isna()]
df[df['form_id'] == 1144]
# 3874 F Square
# 3362 T
# 4651 W
# 226 H Square
# 1144 Unknown
# 8383 W
# 5147 O Square
# 7752 O Square
# 11272 R Square
# 351 P Square
# 338 R Square
# 1813 P Square
# 5949 F Square
# 5179 G Square
# 4980 U Circle
# 303 Q Square
# 2587 K Square
# 6594 R Square
# 756 R Square
# 227 Q Square
# 5146 R Square
# 2985 C 2018
# 476 Q Square
# 482 K Square
# -----------------
# 5913 B
# 3580 N Square


,form_id,contract_num,year_stamp_holo_dc,year_mapping,year_letter
345,1144,2566000861.0,Unknown,2022,U
